# Phase I — YOLO11n object-size behavior (AI-assisted pair)

**Research question:** How does apparent object size influence YOLO11n's detection behavior in VisDrone aerial images? Are smaller objects more likely to be missed? Can similarly sized objects behave differently? Does confidence change with size? Do other factors, such as occlusion, accompany some of the errors?

**Our design:** we compare objects **within each image**, separately for each category, and then look for patterns **across images**. We complement this with an absolute-size view across the whole dataset. This is **VIS4AI Phase I**: we inspect inputs and outputs only, **not** feature maps or internal weights (those are for Phase II).

**Use of Generative AI:** this notebook was developed with Generative AI as a collaborator. We reviewed and tested the code and interpretations ourselves; our key prompts and the suggestions we accepted or rejected are recorded in `phase1.md`.

### Input files

- `no_ai_detections_imgsz-640_conf-0.1.csv`: YOLO11n predictions computed earlier by our teammates (`imgsz=640`, `conf=0.1`). We reuse the predictions, not the visual analysis of the non-AI pair.
- `VisDrone2019-DET-val/annotations/*.txt`: ground-truth labels, needed to identify **missed objects**.
- `VisDrone2019-DET-val/images/*.jpg`: original images for the interactive explorers.

The prediction CSV has `class=99` rows for images where the model returned no predictions. We exclude them from the predictions, but keep those images when counting missed annotated objects.

**Method caveats:**
- We match predictions to ground truth at **IoU ≥ 0.5**. This is our own choice for the analysis and is unrelated to the NMS IoU used during inference (`0.7`). Section 10 checks how sensitive the results are to this threshold.
- Each prediction can be matched only once. Predictions below confidence 0.1 were not saved, so they are unavailable.
- Inference used `imgsz=640`, so the 1920×1080 VisDrone images were **downscaled about 3×** before YOLO saw them. An object of 300 px² in the original image covers only about 33 px² at the model's input resolution. All areas in this notebook are measured in **original image pixels**.

**Categories analysed (6):** VisDrone IDs `1 pedestrian, 2 people → Person`, `3 → Bicycle`, `4 → Car`, `6 → Truck`, `9 → Bus`, `10 motor → Motorcycle`. We do **not** merge *van* into *car* and we leave out *tricycle* and *awning-tricycle*, because COCO has no equivalent classes.

## 1. Load the YOLO predictions

Set `CSV_PATH` (below) and `IMAGE_DIR` / `ANNOTATION_DIR` (Section 2) to match your local folder structure. The paths are relative to the folder where Jupyter was started.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image
import plotly.express as px
from IPython.display import display

CSV_PATH = "no_ai_detections_imgsz-640_conf-0.1.csv"
raw_pred = pd.read_csv(CSV_PATH, sep=";")
required = {"image", "x1", "y1", "x2", "y2", "class", "conf", "width", "height"}
assert required.issubset(raw_pred.columns), f"Missing CSV columns: {required - set(raw_pred.columns)}"

# Class=99 represents NO predictions, not a real detection.
pred = raw_pred[raw_pred["class"] != 99].copy()
pred["class"] = pred["class"].astype(int)
image_sizes = raw_pred.groupby("image")[["width", "height"]].first()
# Process all 548 by default. For a quick first experiment, set MAX_IMAGES = 100.
MAX_IMAGES = None  # e.g. 100; None means all available images
all_image_ids = sorted(raw_pred["image"].unique())[:MAX_IMAGES]

print("Images:", len(all_image_ids))
print("Actual predictions:", len(pred))
print("No-detection images:", raw_pred.loc[raw_pred["class"] == 99, "image"].nunique())
print("Available predicted COCO classes:", pred["class"].value_counts().to_dict())
display(pred.head(3))

Images: 548
Actual predictions: 16051
No-detection images: 17
Available predicted COCO classes: {2: 10897, 0: 3348, 7: 1049, 3: 441, 5: 265, 1: 51}


,image,x1,y1,x2,y2,class,conf,width,height
0,0000001_02999_d_0000005,443.136475,537.388550,502.354614,591.518005,2,0.512813,1920,1080
1,0000001_02999_d_0000005,761.002747,324.441040,824.853516,377.219727,2,0.291104,1920,1080
2,0000001_02999_d_0000005,946.241638,590.175171,1013.095642,690.291016,7,0.290809,1920,1080


## 2. Locate the ground-truth annotation files

**Stop here if the annotation files are missing.** The predictions alone are not enough to identify false negatives. We support two label formats:

1. **Original VisDrone**: comma-separated `x,y,width,height,validity,class,truncation,occlusion` (class IDs **1–10**).
2. **Ultralytics-converted YOLO labels**: space-separated `class,x_center,y_center,width,height` (normalized 0–1, class IDs **0–9**). This version does **not** retain occlusion/truncation.

The code detects the format from the first nonempty annotation line. If your repository has a different folder structure, set `ANNOTATION_DIR` and `IMAGE_DIR` explicitly.

In [2]:
IMAGE_DIR = Path("VisDrone2019-DET-val") / "images"
ANNOTATION_DIR = Path("VisDrone2019-DET-val") / "annotations"

print("Image folder:", IMAGE_DIR)
print("Annotation folder:", ANNOTATION_DIR)

Image folder: VisDrone2019-DET-val\images
Annotation folder: VisDrone2019-DET-val\annotations


## 3. Load the *ground-truth objects* and calculate size

Every row here represents **one real, annotated object**, including those YOLO did not detect. We keep `image`, `class`, `area_px`, and the box coordinates so the results can later be linked back to the aerial image.

`area_px = bounding-box width × bounding-box height`, in square pixels. For our main within-image analysis, we will rank the **same category** by this size; the rank is *relative to the other objects in the same image*, not a global small/medium/large definition.

In [3]:
ORIGINAL_TO_COCO = {
    1: 0,   # Pedestrian -> Person
    2: 0,   # People -> Person
    3: 1,   # Bicycle
    4: 2,   # Car
    6: 7,   # Truck
    9: 5,   # Bus
    10: 3   # Motor -> Motorcycle
}

CONVERTED_TO_COCO = {
    0: 0, 1: 0,
    2: 1, 3: 2,
    5: 7, 8: 5, 9: 3
}

COCO_NAMES = {
    0: "Person",
    1: "Bicycle",
    2: "Car",
    3: "Motorcycle",
    5: "Bus",
    7: "Truck"
}

FOCUS_IDS = set(COCO_NAMES)

def read_ground_truth(image_id):
    path = ANNOTATION_DIR / f"{image_id}.txt"
    if not path.is_file():
        raise FileNotFoundError(f"Missing annotation for image {image_id}: {path}")

    width = int(image_sizes.loc[image_id, "width"])
    height = int(image_sizes.loc[image_id, "height"])
    rows = []
    for line_number, line in enumerate(path.read_text(encoding="utf-8").splitlines(), start=1):
        line = line.strip()
        if not line:
            continue
        original_format = "," in line
        if original_format:
            values = [float(v.strip()) for v in line.rstrip(",").split(",")]
            if len(values) != 8:
                raise ValueError(f"Expected 8 VisDrone values: {path}:{line_number}")
            x, y, bw, bh, validity, original_id, truncation, occlusion = values
            if int(validity) == 0 or int(original_id) not in ORIGINAL_TO_COCO:
                continue
            cls = ORIGINAL_TO_COCO[int(original_id)]
            x1, y1, x2, y2 = x, y, x + bw, y + bh
        else:
            values = [float(v) for v in line.split()]
            if len(values) != 5:
                raise ValueError(f"Expected 5 YOLO label values: {path}:{line_number}")
            converted_id, cx, cy, w_rel, h_rel = values
            if int(converted_id) not in CONVERTED_TO_COCO:
                continue
            cls = CONVERTED_TO_COCO[int(converted_id)]
            x1 = (cx - w_rel / 2) * width
            y1 = (cy - h_rel / 2) * height
            x2 = (cx + w_rel / 2) * width
            y2 = (cy + h_rel / 2) * height
            occlusion, truncation = np.nan, np.nan

        # Clamp to actual image bounds. Prevent out-of-frame/negative boxes.
        x1, x2 = np.clip([x1, x2], 0, width)
        y1, y2 = np.clip([y1, y2], 0, height)
        if x2 <= x1 or y2 <= y1:
            continue
        rows.append({
            "image": image_id, "object_id": f"{image_id}:{line_number}",
            "gt_class": cls, "category": COCO_NAMES[cls],
            "x1": x1, "y1": y1, "x2": x2, "y2": y2,
            "area_px": (x2 - x1) * (y2 - y1),
            "image_width": width, "image_height": height,
            "occlusion": occlusion, "truncation": truncation,
        })
    return rows

all_gt_rows = []
for image_id in all_image_ids:
    all_gt_rows.extend(read_ground_truth(image_id))
gt = pd.DataFrame(all_gt_rows)
if gt.empty:
    raise ValueError("No selected ground-truth annotations found. Check label format and mappings.")
assert set(gt["gt_class"].unique()).issubset(FOCUS_IDS)
print("Images with selected ground-truth objects:", gt["image"].nunique())
print("Number of annotated objects:", len(gt))
display(gt["category"].value_counts().rename_axis("category").to_frame("objects"))
display(gt.head(3))

Images with selected ground-truth objects: 548
Number of annotated objects: 35207


,objects
category,
Car,14064
Person,13969
Motorcycle,4886
Bicycle,1287
Truck,750
Bus,251


,image,object_id,gt_class,category,x1,y1,x2,y2,area_px,image_width,image_height,occlusion,truncation
0,0000001_02999_d_0000005,0000001_02999_d_0000005:1,2,Car,871.0,572.0,925.0,664.0,4968.0,1920,1080,0.0,0.0
1,0000001_02999_d_0000005,0000001_02999_d_0000005:2,2,Car,948.0,592.0,1010.0,684.0,5704.0,1920,1080,0.0,0.0
2,0000001_02999_d_0000005,0000001_02999_d_0000005:3,2,Car,874.0,705.0,941.0,815.0,7370.0,1920,1080,1.0,0.0


In [4]:
def iou_matrix(gt_boxes, pred_boxes):
    if len(gt_boxes) == 0 or len(pred_boxes) == 0:
        return np.zeros((len(gt_boxes), len(pred_boxes)))

    lt = np.maximum(gt_boxes[:, None, :2], pred_boxes[None, :, :2])
    rb = np.minimum(gt_boxes[:, None, 2:], pred_boxes[None, :, 2:])

    wh = np.maximum(rb - lt, 0)
    intersection = wh[:, :, 0] * wh[:, :, 1]

    gt_area = np.prod(gt_boxes[:, 2:] - gt_boxes[:, :2], axis=1)
    pred_area = np.prod(pred_boxes[:, 2:] - pred_boxes[:, :2], axis=1)

    union = gt_area[:, None] + pred_area[None, :] - intersection

    return np.divide(
        intersection, union,
        out=np.zeros_like(intersection),
        where=union > 0
    )

## 4. Match YOLO predictions against ground truth

**Critical stage:** only now can we say whether YOLO detected an object.

We use a *one-to-one* bounding-box match at IoU ≥ 0.5:

- **Detected**: matched prediction with the same mapped category.
- **Wrong class**: matched prediction overlaps the ground truth, but has a different class. This is a **secondary** question, not the main research focus.
- **Missed**: no available prediction matches it.

Correct-class matches are assigned first; unmatched boxes are then checked for cross-category overlaps. This is an **exploratory matching convention**, not a full COCO benchmark evaluator. The CSV may contain detections for bus/truck/motorcycle as well, so those can reveal some cross-class mistakes. It does *not* contain predictions from classes filtered out by your teammate's inference script, so it cannot reveal *every* possible wrong-class prediction.

We do **not** assign a confidence score to a missed object.

In [5]:
MATCH_IOU = 0.50

def match_one_image(gts, preds, iou_thr=MATCH_IOU):
    gts = gts.copy().reset_index(drop=True)
    preds = preds[
        (preds["class"] != 99) &
        (preds["class"].isin(FOCUS_IDS))
    ].reset_index(drop=True)

    gts["status"] = "Missed"

    for col in ["pred_class", "pred_conf", "match_iou",
                "pred_x1", "pred_y1", "pred_x2", "pred_y2"]:
        gts[col] = np.nan

    if gts.empty or preds.empty:
        return gts

    gt_boxes = gts[["x1", "y1", "x2", "y2"]].to_numpy()
    pred_boxes = preds[["x1", "y1", "x2", "y2"]].to_numpy()

    ious = iou_matrix(gt_boxes, pred_boxes)

    pairs = list(map(tuple, np.argwhere(ious >= iou_thr)))

    # Prioritize correct category, then highest IoU
    pairs.sort(key=lambda pair: (
        gts.loc[pair[0], "gt_class"] != preds.loc[pair[1], "class"],
        -ious[pair[0], pair[1]]
    ))

    used_gt, used_pred = set(), set()

    for i, j in pairs:
        if i in used_gt or j in used_pred:
            continue

        used_gt.add(i)
        used_pred.add(j)

        correct = gts.loc[i, "gt_class"] == preds.loc[j, "class"]

        gts.loc[i, "status"] = "Detected" if correct else "Wrong class"
        gts.loc[i, "pred_class"] = preds.loc[j, "class"]
        gts.loc[i, "pred_conf"] = preds.loc[j, "conf"]
        gts.loc[i, "match_iou"] = ious[i, j]

        for coord in ["x1", "y1", "x2", "y2"]:
            gts.loc[i, "pred_" + coord] = preds.loc[j, coord]

    return gts

all_matches = []

for image_id, image_gt in gt.groupby("image"):
    image_pred = pred[pred["image"] == image_id]
    all_matches.append(match_one_image(image_gt, image_pred))

objects = pd.concat(all_matches, ignore_index=True)
objects["detected"] = objects["status"] == "Detected"

print("\nDetection outcomes:")
print(objects["status"].value_counts())

print("\nAvailable categories:")
print(objects["category"].value_counts())

output_file = Path.cwd() / "ai_object_results.csv"
objects.to_csv(output_file, index=False)

print("\nSaved to:", output_file)


Detection outcomes:
status
Missed         26433
Detected        8111
Wrong class      663
Name: count, dtype: int64

Available categories:
category
Car           14064
Person        13969
Motorcycle     4886
Bicycle        1287
Truck           750
Bus             251
Name: count, dtype: int64

Saved to: c:\Users\LENOVO\Documents\JKU\UE AI and Visualization\ai_object_results.csv


## 5. Main comparison: smaller vs larger **within the same image**

For each image **and category**, sort annotated objects by ground-truth area. Compare the **smaller half** to the **larger half**. When there is an odd number, the middle object is not used in this simple split. A group must contain at least two objects. This is not a universal definition of "small"; it compares objects **locally**, as you proposed.

Example: if an image contains five annotated cars, the two smallest cars form "smaller half", the two largest "larger half", and the middle car is excluded from this specific comparison.

We show the number of **objects** and the number of **image-category groups**. We also compare *within-image detection-rate differences* so that one very crowded image does not silently dominate the interpretation.

In [6]:
objects["size_group"] = "Not compared"
objects["within_image_rank"] = np.nan
for (image_id, cls), group in objects.groupby(["image", "gt_class"]):
    indices = group.sort_values(["area_px", "object_id"]).index.to_list()
    n = len(indices)
    if n < 2:
        continue
    half = n // 2
    objects.loc[indices[:half], "size_group"] = "Smaller half"
    objects.loc[indices[-half:], "size_group"] = "Larger half"
    if n % 2:
        objects.loc[indices[half], "size_group"] = "Middle (excluded)"
    objects.loc[indices, "within_image_rank"] = np.arange(1, n + 1)

comp = objects[objects["size_group"].isin(["Smaller half", "Larger half"])].copy()
summary = (comp.groupby(["category", "size_group"], as_index=False)
           .agg(n_objects=("detected", "size"), detection_rate=("detected", "mean"),
                n_images=("image", "nunique")))
print("Object-level overview (not the only visualization):")
display(summary.sort_values(["category", "size_group"]))

fig = px.bar(summary, x="category", y="detection_rate", color="size_group",
             barmode="group", text="n_objects", title="Detected fraction by local size half (within each image)",
             labels={"detection_rate": "Fraction correctly detected", "category": "Ground-truth category"})
fig.update_yaxes(range=[0, 1], tickformat=".0%")
fig.show()

# Matched image/category groups: one comparison = one image and one object category.
paired = (comp.groupby(["image", "category", "size_group"])["detected"]
          .mean().unstack("size_group").dropna().reset_index())
paired["large_minus_small"] = paired["Larger half"] - paired["Smaller half"]
print("Number of image/category comparisons:", len(paired))
print("Comparisons favoring larger objects:", int((paired.large_minus_small > 0).sum()))
print("Comparisons favoring smaller objects:", int((paired.large_minus_small < 0).sum()))
print("Comparisons tied:", int((paired.large_minus_small == 0).sum()))

fig = px.histogram(paired, x="large_minus_small", color="category", barmode="overlay",
                   title="Within-image detection advantage: larger half minus smaller half",
                   labels={"large_minus_small": "Detection-rate difference (larger − smaller)"})
fig.add_vline(x=0, line_dash="dash")
fig.show()

Object-level overview (not the only visualization):


,category,size_group,n_objects,detection_rate,n_images
0,Bicycle,Larger half,531,0.009416,242
1,Bicycle,Smaller half,531,0.001883,242
2,Bus,Larger half,78,0.474359,51
3,Bus,Smaller half,78,0.128205,51
4,Car,Larger half,6898,0.632357,506
5,Car,Smaller half,6898,0.199188,506
6,Motorcycle,Larger half,2308,0.075823,451
7,Motorcycle,Smaller half,2308,0.002600,451
8,Person,Larger half,6847,0.239229,516
9,Person,Smaller half,6847,0.029648,516


Number of image/category comparisons: 1924
Comparisons favoring larger objects: 966
Comparisons favoring smaller objects: 41
Comparisons tied: 917


### Interactive view: every annotated object, not only averages

Each point is one annotated object. The horizontal position is its **size rank within its own image and category**, from 0 (smallest) to 1 (largest), calculated as `(rank − 0.5) / n`. The vertical position separates **correctly detected** objects from objects that were missed or given the wrong class. Hover over a point to see its image name and exact pixel size, then open that image in the image explorer below.

- Image/category groups with **only one object** are excluded, because a single object has no "smaller" or "larger" neighbour.
- Small groups still produce some visible columns of points (e.g. two-object groups always sit at 0.25 and 0.75). This is expected and comes from the ranking, not from the model.
- The black line shows the **detection rate in 10 rank bins**, calculated from *all* eligible objects. The points are a random sample of 4,000, so the chart stays readable.
- The vertical jitter only separates overlapping points visually.

In [7]:
import plotly.graph_objects as go

SAMPLE_SIZE = 4000

group_size = objects.groupby(["image", "gt_class"])["area_px"].transform("size")
scatter_all = objects[group_size >= 2].copy()
scatter_all["n_in_group"] = group_size[group_size >= 2]
rank = scatter_all.groupby(["image", "gt_class"])["area_px"].rank(method="average")
scatter_all["size_rank_in_image"] = (rank - 0.5) / scatter_all["n_in_group"]

# Detection rate per rank bin, from ALL eligible objects (not only the sample)
scatter_all["rank_bin"] = pd.cut(scatter_all["size_rank_in_image"], bins=np.linspace(0, 1, 11), include_lowest=True)
binned = scatter_all.groupby("rank_bin", observed=True)["detected"].mean()
bin_centres = [interval.mid for interval in binned.index]

scatter_objects = scatter_all.sample(n=min(SAMPLE_SIZE, len(scatter_all)), random_state=42)
rng = np.random.default_rng(42)
scatter_objects["detection_y"] = scatter_objects["detected"].astype(float) + rng.normal(0, 0.035, len(scatter_objects))

fig = px.scatter(
    scatter_objects, x="size_rank_in_image", y="detection_y", color="category", opacity=0.5,
    hover_data=["image", "area_px", "status", "n_in_group"],
    title=f"Object size rank vs detection outcome (random sample of {len(scatter_objects):,} of {len(scatter_all):,} objects)",
    labels={"size_rank_in_image": "Relative size rank within image and category (0 = smallest, 1 = largest)",
            "detection_y": "Detection outcome"},
    render_mode="webgl",
)
fig.add_trace(go.Scatter(x=bin_centres, y=binned.values, mode="lines+markers", name="Detection rate (all objects)",
                         line=dict(color="black", width=3)))
fig.update_yaxes(tickvals=[0, 0.25, 0.5, 0.75, 1], ticktext=["Missed / Wrong class", "25%", "50%", "75%", "Detected"],
                 range=[-0.2, 1.2])
fig.show()

print("Objects excluded (only object of its category in its image):", int((group_size < 2).sum()))
print("Detection rate by rank bin:")
display(binned.round(3))

Objects excluded (only object of its category in its image): 368
Detection rate by rank bin:


rank_bin
(-0.001, 0.1]    0.043
(0.1, 0.2]       0.063
(0.2, 0.3]       0.097
(0.3, 0.4]       0.114
(0.4, 0.5]       0.172
(0.5, 0.6]       0.228
(0.6, 0.7]       0.286
(0.7, 0.8]       0.359
(0.8, 0.9]       0.432
(0.9, 1.0]       0.542
Name: detected, dtype: float64

## 6. Do smaller *correctly detected* objects have lower confidence?

Confidence only exists for a **prediction**. We therefore investigate model confidence for **correctly matched detections**, and analyze missed objects separately. We cannot pretend missed objects have zero confidence.

Use color to compare categories. For this chart we use the **ground-truth box size**, so the measure is consistent across detections and missed objects.


Note: predictions below confidence 0.1 were not saved, so the scatter has a hard floor at 0.1. A detection that YOLO made with lower confidence appears in this notebook as a missed object.

In [8]:
matched = objects[objects["status"] == "Detected"].copy()
if matched.empty:
    print("No correctly matched predictions to plot.")
else:
    fig = px.scatter(
        matched, x="area_px", y="pred_conf", color="category",
        log_x=True, opacity=0.45,
        hover_data=["image", "size_group", "match_iou"],
        title="Correct detections: object area vs YOLO confidence",
        labels={"area_px": "Ground-truth area (pixels², logarithmic)",
                "pred_conf": "Prediction confidence"}
    )
    fig.show()
    print("Spearman rank correlation (area vs confidence), exploratory:")
    display(matched.groupby("category")[["area_px", "pred_conf"]]
            .corr(method="spearman").loc[(slice(None), "area_px"), "pred_conf"])

Spearman rank correlation (area vs confidence), exploratory:


category           
Bicycle     area_px    0.885714
Bus         area_px    0.636580
Car         area_px    0.414024
Motorcycle  area_px    0.099286
Person      area_px    0.375474
Truck       area_px    0.429535
Name: pred_conf, dtype: float64

## 7. Counterexamples: similarly sized objects, opposite outcomes

This is a deeper investigation than a benchmark score. Within each image and class, find a **missed** object whose area differs by no more than 20% from a **detected** object. These are particularly interesting: size alone cannot explain why these objects behaved differently.

Inspect the images: are the objects occluded, near each other, or surrounded by other objects? **These are hypotheses**, not conclusions. You can change `SIMILARITY_RATIO` and see what changes.

In [9]:
SIMILARITY_RATIO = 1.20  # up to 20% larger/smaller area
pairs = []
for (image_id, cat), g in objects.groupby(["image", "category"]):
    misses = g[g["status"] == "Missed"]
    detects = g[g["status"] == "Detected"]
    if misses.empty or detects.empty:
        continue
    for _, miss in misses.iterrows():
        ratios = np.maximum(detects["area_px"].to_numpy() / miss.area_px,
                            miss.area_px / detects["area_px"].to_numpy())
        best_idx = int(np.argmin(ratios))
        if ratios[best_idx] <= SIMILARITY_RATIO:
            good = detects.iloc[best_idx]
            pairs.append({"image": image_id, "category": cat,
                          "missed_id": miss.object_id, "missed_area": miss.area_px,
                          "detected_id": good.object_id, "detected_area": good.area_px,
                          "size_ratio": ratios[best_idx]})

counterexamples = pd.DataFrame(pairs, columns=[
    "image", "category", "missed_id", "missed_area", "detected_id", "detected_area", "size_ratio"])
print("Similar-size detected-versus-missed comparisons:", len(counterexamples))
print("Images containing these counterexamples:", counterexamples.image.nunique())
if not counterexamples.empty:
    display(counterexamples.sort_values("size_ratio").head(15))

Similar-size detected-versus-missed comparisons: 6118
Images containing these counterexamples: 476


,image,category,missed_id,missed_area,detected_id,detected_area,size_ratio
2710,0000280_02201_d_0000623,Car,0000280_02201_d_0000623:34,3250.0,0000280_02201_d_0000623:43,3250.0,1.0
5250,0000330_04001_d_0000820,Person,0000330_04001_d_0000820:32,540.0,0000330_04001_d_0000820:7,540.0,1.0
646,0000086_01954_d_0000005,Person,0000086_01954_d_0000005:144,456.0,0000086_01954_d_0000005:44,456.0,1.0
3879,0000291_04001_d_0000888,Person,0000291_04001_d_0000888:88,702.0,0000291_04001_d_0000888:22,702.0,1.0
638,0000086_01954_d_0000005,Person,0000086_01954_d_0000005:66,385.0,0000086_01954_d_0000005:147,385.0,1.0
157,0000001_08414_d_0000013,Person,0000001_08414_d_0000013:105,374.0,0000001_08414_d_0000013:49,374.0,1.0
637,0000086_01954_d_0000005,Person,0000086_01954_d_0000005:65,480.0,0000086_01954_d_0000005:13,480.0,1.0
3875,0000291_04001_d_0000888,Person,0000291_04001_d_0000888:83,1078.0,0000291_04001_d_0000888:3,1078.0,1.0
3868,0000291_04001_d_0000888,Person,0000291_04001_d_0000888:40,702.0,0000291_04001_d_0000888:22,702.0,1.0
1736,0000249_01514_d_0000005,Car,0000249_01514_d_0000005:27,300.0,0000249_01514_d_0000005:25,300.0,1.0


## 8. Absolute size: detection rate across the whole dataset

The within-image comparison (Section 5) controls for altitude and scene, but it does not tell us *how large* an object must be for YOLO to find it. Here we group all annotated objects by their **absolute** bounding-box area and plot the detection rate per size bin.

The second line of each x-axis label gives the approximate side length the object has **at the model's input resolution** (`imgsz=640`, i.e. about 3× smaller than the original image). For comparison, COCO calls an object "small" below 32 × 32 px. Bins with fewer than 30 objects of a category are hidden, because their rates would be unreliable.

In [10]:
SIZE_BINS = [0, 256, 512, 1024, 2048, 4096, 9216, np.inf]
MIN_BIN_COUNT = 30
scale = 640 / objects[["image_width", "image_height"]].max(axis=1)   # YOLO letterbox scale per image
objects["input_side_px"] = np.sqrt(objects["area_px"]) * scale

def bin_label(lo, hi):
    side = lambda a: np.sqrt(a) * 640 / 1920
    if np.isinf(hi):
        return f">{lo:,.0f} px²<br>(>{side(lo):.0f} px at input)"
    return f"{lo:,.0f}–{hi:,.0f} px²<br>(≈{side(lo):.0f}–{side(hi):.0f} px at input)"

labels = [bin_label(lo, hi) for lo, hi in zip(SIZE_BINS[:-1], SIZE_BINS[1:])]
objects["size_bin"] = pd.cut(objects["area_px"], SIZE_BINS, labels=labels, right=True)

by_cat = (objects.groupby(["category", "size_bin"], observed=False)["detected"]
          .agg(detection_rate="mean", n_objects="size").reset_index())
overall = (objects.groupby("size_bin", observed=False)["detected"]
           .agg(detection_rate="mean", n_objects="size").reset_index().assign(category="All categories"))
size_curve = pd.concat([by_cat, overall], ignore_index=True)
size_curve.loc[size_curve["n_objects"] < MIN_BIN_COUNT, "detection_rate"] = np.nan

fig = px.line(size_curve, x="size_bin", y="detection_rate", color="category", markers=True,
              hover_data=["n_objects"], category_orders={"size_bin": labels},
              title="Detection rate by absolute object size (original-image pixels)",
              labels={"size_bin": "Ground-truth bounding-box area", "detection_rate": "Fraction correctly detected"})
fig.update_traces(selector=dict(name="All categories"), line=dict(color="black", width=4))
fig.update_yaxes(range=[0, 1], tickformat=".0%")
fig.show()

display(overall[["size_bin", "n_objects", "detection_rate"]].round(3))

,size_bin,n_objects,detection_rate
0,0–256 px²<br>(≈0–5 px at input),11367,0.013
1,256–512 px²<br>(≈5–8 px at input),6828,0.096
2,"512–1,024 px²<br>(≈8–11 px at input)",6530,0.233
3,"1,024–2,048 px²<br>(≈11–15 px at input)",4955,0.391
4,"2,048–4,096 px²<br>(≈15–21 px at input)",2781,0.608
5,"4,096–9,216 px²<br>(≈21–32 px at input)",1811,0.765
6,">9,216 px²<br>(>32 px at input)",935,0.834


## 9. Occlusion: does it explain errors beyond size?

VisDrone annotates the occlusion of each object: **0 = none**, **1 = partial (1–50 %)**, **2 = heavy (50–100 %)**. Occluded objects also tend to look smaller, because part of the object is hidden. So we compare occlusion levels **within each size bin**: if the lines stay apart, occlusion matters independently of size.

We then go back to the similar-size counterexamples from Section 7 and ask whether the *missed* object of each pair is more often occluded than its *detected* partner.

In [11]:
OCCLUSION_NAMES = {0: "0 none", 1: "1 partial", 2: "2 heavy"}
occ = objects.dropna(subset=["occlusion"]).copy()
occ["occlusion_level"] = occ["occlusion"].astype(int).map(OCCLUSION_NAMES)

display(occ.groupby("occlusion_level")["detected"].agg(n_objects="size", detection_rate="mean").round(3))

occ_curve = (occ.groupby(["occlusion_level", "size_bin"], observed=False)["detected"]
             .agg(detection_rate="mean", n_objects="size").reset_index())
occ_curve.loc[occ_curve["n_objects"] < MIN_BIN_COUNT, "detection_rate"] = np.nan
fig = px.line(occ_curve, x="size_bin", y="detection_rate", color="occlusion_level", markers=True,
              hover_data=["n_objects"], category_orders={"size_bin": labels},
              title="Detection rate by size, separately for each occlusion level",
              labels={"size_bin": "Ground-truth bounding-box area", "detection_rate": "Fraction correctly detected",
                      "occlusion_level": "Occlusion"})
fig.update_yaxes(range=[0, 1], tickformat=".0%")
fig.show()

# Counterexample pairs: occlusion of the missed object vs its similar-size detected partner
if not counterexamples.empty and objects["occlusion"].notna().any():
    occ_by_id = objects.set_index("object_id")["occlusion"]
    pair_occ = counterexamples.assign(missed_occ=counterexamples["missed_id"].map(occ_by_id),
                                      detected_occ=counterexamples["detected_id"].map(occ_by_id))
    print(f"Counterexample pairs: {len(pair_occ)}")
    print("Missed object MORE occluded than its detected partner:", int((pair_occ.missed_occ > pair_occ.detected_occ).sum()))
    print("Missed object LESS occluded:", int((pair_occ.missed_occ < pair_occ.detected_occ).sum()))
    print("Same occlusion level:", int((pair_occ.missed_occ == pair_occ.detected_occ).sum()))

,n_objects,detection_rate
occlusion_level,,
0 none,15401,0.329
1 partial,16975,0.173
2 heavy,2831,0.041


Counterexample pairs: 6118
Missed object MORE occluded than its detected partner: 2588
Missed object LESS occluded: 701
Same occlusion level: 2829


## 10. Sensitivity check: does the IoU threshold create the size effect?

IoU is harsh on small boxes: a prediction that is off by 3 pixels barely changes the IoU of a large car, but can push a tiny pedestrian below 0.5. Part of the "small objects are missed" pattern could therefore come from **our matching rule**, not from the model. We re-run the matching at **IoU ≥ 0.3** and compare.

If small objects are still detected much less often at 0.3, the size effect is real model behaviour. If the gap shrinks a lot, YOLO often *finds* small objects but **localises** them imprecisely, which is a different kind of failure.

In [12]:
SENSITIVITY_IOUS = [0.3, 0.5]
sens_rows = []
for thr in SENSITIVITY_IOUS:
    res = pd.concat([match_one_image(g, pred[pred["image"] == image_id], iou_thr=thr)
                     for image_id, g in gt.groupby("image")], ignore_index=True)
    res["detected"] = res["status"] == "Detected"
    res["size_bin"] = pd.cut(res["area_px"], SIZE_BINS, labels=labels)
    rates = res.groupby("size_bin", observed=False)["detected"].agg(detection_rate="mean", n_objects="size").reset_index()
    sens_rows.append(rates.assign(iou_threshold=f"IoU ≥ {thr}"))
    print(f"IoU ≥ {thr}: overall detection rate {res['detected'].mean():.1%}")

sens = pd.concat(sens_rows, ignore_index=True)
fig = px.line(sens, x="size_bin", y="detection_rate", color="iou_threshold", markers=True,
              hover_data=["n_objects"], category_orders={"size_bin": labels},
              title="Detection rate by size at two IoU matching thresholds",
              labels={"size_bin": "Ground-truth bounding-box area", "detection_rate": "Fraction correctly detected",
                      "iou_threshold": "Matching rule"})
fig.update_yaxes(range=[0, 1], tickformat=".0%")
fig.show()
display(sens.pivot(index="size_bin", columns="iou_threshold", values="detection_rate").round(3))

IoU ≥ 0.3: overall detection rate 25.6%
IoU ≥ 0.5: overall detection rate 23.0%


iou_threshold,IoU ≥ 0.3,IoU ≥ 0.5
size_bin,,
0–256 px²<br>(≈0–5 px at input),0.021,0.013
256–512 px²<br>(≈5–8 px at input),0.125,0.096
"512–1,024 px²<br>(≈8–11 px at input)",0.272,0.233
"1,024–2,048 px²<br>(≈11–15 px at input)",0.427,0.391
"2,048–4,096 px²<br>(≈15–21 px at input)",0.644,0.608
"4,096–9,216 px²<br>(≈21–32 px at input)",0.798,0.765
">9,216 px²<br>(>32 px at input)",0.855,0.834


## Image explorer: which objects in one image were detected?

Pick an image and a category. Left: annotated objects, coloured **green = detected**, **red = missed**, **orange = wrong class**, and numbered from smallest to largest. Right: the same objects ranked by size. Use the hover information from the charts above to choose images worth inspecting.

In [13]:
import ipywidgets as widgets

COLORS = {
    "Detected": "green",
    "Missed": "red",
    "Wrong class": "orange"
}

def explore_image(image_id, category):

    data = objects[
        (objects["image"] == image_id) &
        (objects["category"] == category)
    ].copy()

    data = data.sort_values("area_px").reset_index(drop=True)
    image_path = Path(IMAGE_DIR) / f"{image_id}.jpg"

    if not image_path.exists():
        print("Image not found:", image_path)
        return

    img = Image.open(image_path).convert("RGB")

    fig, (ax1, ax2) = plt.subplots(
        1, 2, figsize=(15, 6)
    )
    
    ax1.imshow(img)

    for i, row in data.iterrows():
        color = COLORS.get(row["status"], "gray")

        rect = Rectangle(
            (row["x1"], row["y1"]),
            row["x2"] - row["x1"],
            row["y2"] - row["y1"],
            fill=False,
            edgecolor=color,
            linewidth=2
        )

        ax1.add_patch(rect)

        ax1.text(
            row["x1"], row["y1"],
            str(i + 1),
            color="white",
            fontsize=8,
            bbox=dict(facecolor=color, pad=1)
        )

    ax1.set_title(f"Image: {image_id}")
    ax1.axis("off")

    ranks = np.arange(1, len(data) + 1)

    ax2.scatter(
        ranks,
        data["area_px"],
        c=data["status"].map(COLORS).fillna("gray"),
        s=70
    )

    ax2.set_xlabel("Objects ranked smallest to largest")
    ax2.set_ylabel("Bounding-box area (pixels²)")
    ax2.set_title(f"{category}: Size vs Detection")
    ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

    print("Detection outcomes:")
    print(data["status"].value_counts())

widgets.interact(
    explore_image,
    image_id=widgets.Dropdown(
        options=sorted(objects["image"].unique()),
        description="Image:"
    ),
    category=widgets.Dropdown(
        options=sorted(objects["category"].unique()),
        description="Category:"
    )
)

interactive(children=(Dropdown(description='Image:', options=('0000001_02999_d_0000005', '0000001_03499_d_0000…

<function __main__.explore_image(image_id, category)>

## Confidence explorer: what did YOLO predict, and how confident was it?

This view starts from the **predictions** rather than the ground truth. Every predicted box of the selected category is labelled with its confidence and coloured by its outcome:

- **green**: correct, i.e. it matches an annotated object of the same category (IoU ≥ 0.5)
- **orange**: wrong class, i.e. it overlaps an annotated object of a *different* category
- **purple**: false positive, i.e. there is no annotated object of an analysed category underneath. Some of these may be real objects that were left out of our ground truth, such as **vans** predicted as *car*, so inspect them before calling them errors.

Missed annotated objects of the selected category are drawn as **thin red dashed boxes**. They have no confidence value, because YOLO produced no matching prediction for them.

In [14]:
detections = pred[pred["class"].isin(FOCUS_IDS)].copy()
detections["area_px"] = (detections["x2"] - detections["x1"]) * (detections["y2"] - detections["y1"])

# Label each prediction using the matching results in `objects`
COORDS = ["x1", "y1", "x2", "y2"]
matched_preds = objects.loc[objects["status"].isin(["Detected", "Wrong class"]),
                            ["image", "status"] + ["pred_" + c for c in COORDS]].copy()
matched_preds.columns = ["image", "pred_outcome"] + COORDS
for frame in (detections, matched_preds):
    frame[COORDS] = frame[COORDS].round(2)
detections = detections.merge(matched_preds, on=["image"] + COORDS, how="left")
detections["pred_outcome"] = (detections["pred_outcome"]
                              .replace({"Detected": "Correct"})
                              .fillna("False positive"))

PRED_COLORS = {"Correct": "green", "Wrong class": "orange", "False positive": "purple"}
print("Prediction outcomes:")
display(detections["pred_outcome"].value_counts())


def explore_confidence(image_id, category):
    class_id = next(k for k, v in COCO_NAMES.items() if v == category)
    data = detections[(detections["image"] == image_id) & (detections["class"] == class_id)].copy()
    missed = objects[(objects["image"] == image_id) & (objects["gt_class"] == class_id)
                     & (objects["status"] != "Detected")]

    if data.empty and missed.empty:
        print("No predictions or annotated objects of this category in this image.")
        return

    data = data.sort_values("area_px").reset_index(drop=True)
    img = Image.open(Path(IMAGE_DIR) / f"{image_id}.jpg").convert("RGB")
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    ax1.imshow(img)

    for _, row in missed.iterrows():
        ax1.add_patch(Rectangle((row["x1"], row["y1"]), row["x2"] - row["x1"], row["y2"] - row["y1"],
                                fill=False, edgecolor="red", linewidth=1, linestyle="--"))

    for i, row in data.iterrows():
        color = PRED_COLORS[row["pred_outcome"]]
        ax1.add_patch(Rectangle((row["x1"], row["y1"]), row["x2"] - row["x1"], row["y2"] - row["y1"],
                                fill=False, edgecolor=color, linewidth=2))
        ax1.text(row["x1"], max(0, row["y1"] - 5), f"{i + 1}: {row['conf']:.2f}",
                 color="white", fontsize=8, bbox=dict(facecolor=color, alpha=0.8, pad=1))

    ax1.set_title(f"{category} — {image_id}  (red dashed = missed annotated object)")
    ax1.axis("off")

    for outcome, color in PRED_COLORS.items():
        subset = data[data["pred_outcome"] == outcome]
        ax2.scatter(subset["area_px"], subset["conf"], c=color, s=60, label=f"{outcome} ({len(subset)})")
    for i, row in data.iterrows():
        ax2.annotate(str(i + 1), (row["area_px"], row["conf"]), xytext=(4, 4), textcoords="offset points")

    ax2.set_xlabel("Predicted box area (pixels²)")
    ax2.set_ylabel("YOLO confidence")
    ax2.set_title(f"Predicted {category}: size vs confidence  |  missed annotated objects: {len(missed)}")
    ax2.set_ylim(0, 1)
    ax2.grid(alpha=0.3)
    ax2.legend(loc="upper left")
    plt.tight_layout()
    plt.show()


widgets.interact(
    explore_confidence,
    image_id=widgets.Dropdown(options=sorted(objects["image"].unique()), description="Image:"),
    category=widgets.Dropdown(options=sorted(COCO_NAMES.values()), description="Category:"),
)

Prediction outcomes:


pred_outcome
Correct           8111
False positive    7277
Wrong class        663
Name: count, dtype: int64

interactive(children=(Dropdown(description='Image:', options=('0000001_02999_d_0000005', '0000001_03499_d_0000…

<function __main__.explore_confidence(image_id, category)>

In [15]:
RESULT_PATH = Path.cwd() / "ai_object_results.csv"
objects.to_csv(RESULT_PATH, index=False)